# Multi-Model Ensemble Stacking for Virality Prediction

This notebook implements a stacking ensemble that combines:
- **Spotify XGBoost** (RandomizedSearch-tuned)
- **Spotify RandomForest** (GridSearch-tuned)
- **YouTube LGBM** (pre-trained model)

The ensemble learns optimal weights to combine predictions from all three base models for improved virality prediction.


## 1. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from pathlib import Path

# ML and ensemble methods
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, roc_auc_score, confusion_matrix, 
                            classification_report, roc_curve)
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# HuggingFace Hub for model download
from huggingface_hub import hf_hub_download
import requests  # For fallback download method

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


In [2]:
# SSL Certificate Fix (run this first if you get SSL errors)
import ssl
import certifi
import urllib3

# Fix SSL verification
ssl_context = ssl.create_default_context(cafile=certifi.where())
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configure requests library
import requests
requests.packages.urllib3.util.ssl_.create_urllib3_context = lambda: ssl_context

print("✓ SSL certificates configured")


✓ SSL certificates configured


## 2. Load Pre-trained Models from Hugging Face

In [3]:
# Configure HuggingFace Hub with SSL fix for certificate issues
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Set environment variable to handle SSL issues
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# Download XGBoost model from HuggingFace
print("Downloading Spotify XGBoost model (RandomizedSearch-tuned)...")
xgb_model = None
try:
    xgb_model_path = hf_hub_download(
        repo_id="vancenceho/vcp-spotify-xgb-randomizedsearch",
        filename="xgb_tuned_randomized.joblib",
        cache_dir="models"
    )
    xgb_model = joblib.load(xgb_model_path)
    print(f"✓ XGBoost model loaded from: {xgb_model_path}")
except Exception as e:
    print(f"⚠ Error downloading XGBoost model: {e}")
    print("\nTrying alternative download method (ignoring SSL verification)...")
    try:
        # Alternative: Use requests with SSL verification disabled
        import requests
        from huggingface_hub import file_download
        url = "https://huggingface.co/vancenceho/vcp-spotify-xgb-randomizedsearch/resolve/main/xgb_tuned_randomized.joblib"
        print(f"Downloading from: {url}")
        response = requests.get(url, verify=False, timeout=30)
        response.raise_for_status()
        os.makedirs("models", exist_ok=True)
        model_path = "models/xgb_tuned_randomized.joblib"
        with open(model_path, 'wb') as f:
            f.write(response.content)
        xgb_model = joblib.load(model_path)
        print(f"✓ XGBoost model downloaded successfully")
    except Exception as e2:
        print(f"✗ Alternative download also failed: {e2}")
        print("\nFix: Run this command in terminal:")
        print("  pip install certifi")
        print("  python -m certifi (to get certificate path)")

# Download RandomForest model from HuggingFace
print("\nDownloading Spotify RandomForest model (GridSearch-tuned)...")
rf_model = None
try:
    rf_model_path = hf_hub_download(
        repo_id="vancenceho/vcp-spotify-rf-gridsearch",
        filename="rf_tuned_gridsearch.joblib",
        cache_dir="models"
    )
    rf_model = joblib.load(rf_model_path)
    print(f"✓ RandomForest model loaded from: {rf_model_path}")
except Exception as e:
    print(f"⚠ Error downloading RandomForest model: {e}")
    print("\nTrying alternative download method (ignoring SSL verification)...")
    try:
        import requests
        url = "https://huggingface.co/vancenceho/vcp-spotify-rf-gridsearch/resolve/main/rf_tuned_gridsearch.joblib"
        print(f"Downloading from: {url}")
        response = requests.get(url, verify=False, timeout=30)
        response.raise_for_status()
        os.makedirs("models", exist_ok=True)
        model_path = "models/rf_tuned_gridsearch.joblib"
        with open(model_path, 'wb') as f:
            f.write(response.content)
        rf_model = joblib.load(model_path)
        print(f"✓ RandomForest model downloaded successfully")
    except Exception as e2:
        print(f"✗ Alternative download also failed: {e2}")
        print("\nFix: Run this command in terminal:")
        print("  pip install certifi")
        print("  python -m certifi (to get certificate path)")


'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/vancenceho/vcp-spotify-xgb-randomizedsearch/resolve/main/xgb_tuned_randomized.joblib
Retrying in 1s [Retry 1/5].


⚠ Error downloading XGBoost model: Cannot send a request, as the client has been closed.

Trying alternative download method (ignoring SSL verification)...
✓ XGBoost model downloaded successfully



'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)' thrown while requesting HEAD https://huggingface.co/vancenceho/vcp-spotify-rf-gridsearch/resolve/main/rf_tuned_gridsearch.joblib
Retrying in 1s [Retry 1/5].


⚠ Error downloading RandomForest model: Cannot send a request, as the client has been closed.

Trying alternative download method (ignoring SSL verification)...
✓ RandomForest model downloaded successfully


## 3. Load YouTube Dataset Model

In [4]:
# Load YouTube LGBM model (from 02_youtube_model_baseline.ipynb)
print("Loading YouTube LGBM model...")
youtube_model_path = "models/lgbm_randomized.joblib"

if os.path.exists(youtube_model_path):
    youtube_model = joblib.load(youtube_model_path)
    print(f"✓ YouTube LGBM model loaded from: {youtube_model_path}")
else:
    print(f"⚠ YouTube model not found at {youtube_model_path}")
    print("You can train this model by running 02_youtube_model_baseline.ipynb")

print("\n" + "="*60)
print("MODELS LOADED:")
print("="*60)
print("1. XGBoost (Spotify) - RandomizedSearch tuned")
print("2. RandomForest (Spotify) - GridSearch tuned")
print("3. LGBM (YouTube) - RandomizedSearch tuned")
print("="*60)


Loading YouTube LGBM model...
✓ YouTube LGBM model loaded from: models/lgbm_randomized.joblib

MODELS LOADED:
1. XGBoost (Spotify) - RandomizedSearch tuned
2. RandomForest (Spotify) - GridSearch tuned
3. LGBM (YouTube) - RandomizedSearch tuned


## 4. Prepare Data for Ensemble

In [ ]:
# Load YouTube dataset and features (from 02_youtube_model_baseline.ipynb)
print("Loading YouTube dataset...")

DATA_PATH = "data/processed"
df = pd.read_csv(f"{DATA_PATH}/youtube_features_cleaned.csv")

# Assume target variable name - adjust if different
target_col = 'viral'  # or 'is_viral' depending on your dataset
if target_col not in df.columns:
    # Try to find the target column
    possible_targets = [col for col in df.columns if 'viral' in col.lower()]
    if possible_targets:
        target_col = possible_targets[0]
        print(f"Found target column: {target_col}")
    else:
        print("Warning: Could not find target column. Assuming last column is target.")
        target_col = df.columns[-1]

# Separate features and target
engagement_cols = [col for col in df.columns if 'engagement' in col.lower() or 'view' in col.lower()]
feature_cols = [col for col in df.columns if col != target_col and col not in engagement_cols]

# For demo: use a subset of features that are common across datasets
X = df[feature_cols].fillna(0)
y = df[target_col]

print(f"Dataset shape: {X.shape}")
print(f"Target distribution:")
print(y.value_counts())

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


## 5. Implement Stacking Ensemble

In [ ]:
# Create a wrapper for models that may use different feature sets
class ModelWrapper:
    """Wrapper to handle models with potentially different feature sets"""
    def __init__(self, model, name):
        self.model = model
        self.name = name
    
    def fit(self, X, y):
        try:
            self.model.fit(X, y)
        except Exception as e:
            print(f"Warning: {self.name} fit failed - {e}")
        return self
    
    def predict(self, X):
        try:
            return self.model.predict(X)
        except Exception as e:
            print(f"Warning: {self.name} predict failed - {e}")
            return np.zeros(len(X), dtype=int)
    
    def predict_proba(self, X):
        if hasattr(self.model, 'predict_proba'):
            try:
                return self.model.predict_proba(X)
            except Exception as e:
                print(f"Warning: {self.name} predict_proba failed - {e}")
                proba = self.predict(X)
                return np.column_stack([1-proba, proba])
        else:
            proba = self.predict(X)
            return np.column_stack([1-proba, proba])

# Create base models for stacking
print("Creating base models for stacking ensemble...")

# Recreate models as base learners if needed
base_models = [
    ('xgb', ModelWrapper(xgb_model, 'XGBoost')),
    ('rf', ModelWrapper(rf_model, 'RandomForest')),
    ('lgbm', ModelWrapper(youtube_model, 'LGBM'))
]

# Create meta-learner (Logistic Regression is simple and effective for stacking)
meta_learner = LogisticRegression(random_state=42, max_iter=1000)

# Create stacking classifier
stacking_ensemble = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_learner,
    cv=5,  # 5-fold cross-validation for generating meta-features
    stack_method='predict_proba'  # Use probability predictions from base models
)

print("✓ Stacking ensemble created with:")
print(f"  - Base models: {len(base_models)}")
print(f"  - Meta-learner: Logistic Regression")
print(f"  - Cross-validation folds: 5")


## 6. Train Meta-learner

In [ ]:
# Train the stacking ensemble
print("Training stacking ensemble on training data...")
print("(This may take a few minutes due to cross-validation)...\n")

try:
    stacking_ensemble.fit(X_train, y_train)
    print("✓ Stacking ensemble training completed!")
except Exception as e:
    print(f"⚠ Training error: {e}")
    print("Attempting to fit without cross-validation...")
    # Simpler approach without stacking if there are issues
    stacking_ensemble = LogisticRegression(random_state=42, max_iter=1000)
    stacking_ensemble.fit(X_train, y_train)

# Get meta-learner weights (if available)
if hasattr(stacking_ensemble, 'final_estimator_'):
    print("\nMeta-learner (Logistic Regression) coefficients:")
    coef = stacking_ensemble.final_estimator_.coef_[0]
    base_names = ['XGBoost', 'RandomForest', 'LGBM (YouTube)', 'XGBoost_class1', 'RF_class1', 'LGBM_class1']
    for name, weight in zip(base_names[:len(coef)], coef):
        print(f"  {name}: {weight:.4f}")


## 7. Evaluate Ensemble Performance

In [ ]:
# Make predictions with ensemble
print("Generating predictions with stacking ensemble...\n")

ensemble_pred = stacking_ensemble.predict(X_test)
ensemble_pred_proba = stacking_ensemble.predict_proba(X_test)[:, 1]

# Calculate metrics
ensemble_accuracy = accuracy_score(y_test, ensemble_pred)
ensemble_precision = precision_score(y_test, ensemble_pred)
ensemble_recall = recall_score(y_test, ensemble_pred)
ensemble_f1 = f1_score(y_test, ensemble_pred)
ensemble_auc = roc_auc_score(y_test, ensemble_pred_proba)

# Get individual model predictions for comparison
print("="*70)
print("ENSEMBLE AND BASE MODEL PERFORMANCE COMPARISON")
print("="*70)

results = []

# Ensemble results
results.append({
    'Model': 'STACKING ENSEMBLE',
    'Accuracy': ensemble_accuracy,
    'Precision': ensemble_precision,
    'Recall': ensemble_recall,
    'F1-Score': ensemble_f1,
    'ROC-AUC': ensemble_auc
})

# Base model results
for name, model_wrapper in base_models:
    try:
        pred = model_wrapper.predict(X_test)
        pred_proba = model_wrapper.predict_proba(X_test)[:, 1]
        
        acc = accuracy_score(y_test, pred)
        prec = precision_score(y_test, pred)
        rec = recall_score(y_test, pred)
        f1 = f1_score(y_test, pred)
        auc = roc_auc_score(y_test, pred_proba)
        
        results.append({
            'Model': model_wrapper.name.upper(),
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1,
            'ROC-AUC': auc
        })
    except Exception as e:
        print(f"Warning: Could not evaluate {model_wrapper.name} - {e}")

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print("="*70)

# Detailed classification report for ensemble
print("\nDetailed Classification Report (STACKING ENSEMBLE):")
print(classification_report(y_test, ensemble_pred, target_names=['Not Viral', 'Viral']))

# Confusion matrix
cm_ensemble = confusion_matrix(y_test, ensemble_pred)
print(f"\nConfusion Matrix:")
print(cm_ensemble)


## 8. Make Virality Predictions with Ensemble

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Model Performance Comparison
ax1 = axes[0, 0]
models = results_df['Model'].values
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(models))
width = 0.2

for i, metric in enumerate(metrics):
    values = results_df[metric].values
    ax1.bar(x + i*width, values, width, label=metric)

ax1.set_ylabel('Score', fontsize=11)
ax1.set_title('Model Performance Comparison', fontsize=12, fontweight='bold')
ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels(models, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0, 1])

# 2. ROC Curve Comparison
ax2 = axes[0, 1]
fpr_ensemble, tpr_ensemble, _ = roc_curve(y_test, ensemble_pred_proba)
ax2.plot(fpr_ensemble, tpr_ensemble, linewidth=2.5, label=f'Ensemble (AUC={ensemble_auc:.3f})')

# Add base models' ROC curves
colors = ['orange', 'green', 'red']
for (name, model_wrapper), color in zip(base_models, colors):
    try:
        pred_proba = model_wrapper.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, pred_proba)
        auc = roc_auc_score(y_test, pred_proba)
        ax2.plot(fpr, tpr, color=color, alpha=0.6, linewidth=1.5, label=f'{model_wrapper.name} (AUC={auc:.3f})')
    except:
        pass

ax2.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('ROC Curves - Ensemble vs Base Models', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(alpha=0.3)

# 3. Confusion Matrix Heatmap
ax3 = axes[1, 0]
sns.heatmap(cm_ensemble, annot=True, fmt='d', cmap='Blues', ax=ax3, cbar=False)
ax3.set_title('Confusion Matrix - Stacking Ensemble', fontsize=12, fontweight='bold')
ax3.set_ylabel('True Label')
ax3.set_xlabel('Predicted Label')
ax3.set_yticklabels(['Not Viral', 'Viral'], rotation=0)
ax3.set_xticklabels(['Not Viral', 'Viral'], rotation=0)

# 4. Prediction Confidence Distribution
ax4 = axes[1, 1]
viral_proba = ensemble_pred_proba[y_test == 1]
non_viral_proba = ensemble_pred_proba[y_test == 0]
ax4.hist(non_viral_proba, bins=30, alpha=0.6, label='Not Viral (True)', color='blue')
ax4.hist(viral_proba, bins=30, alpha=0.6, label='Viral (True)', color='red')
ax4.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision Threshold')
ax4.set_xlabel('Ensemble Virality Probability', fontsize=11)
ax4.set_ylabel('Frequency', fontsize=11)
ax4.set_title('Prediction Confidence Distribution', fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visualizations complete!")


## 9. Save Ensemble & Generate Predictions on New Data

In [ ]:
# Save the ensemble model
import os
os.makedirs("models", exist_ok=True)

ensemble_save_path = "models/ensemble_stacking_virality.joblib"
joblib.dump(stacking_ensemble, ensemble_save_path)
print(f"✓ Ensemble model saved to: {ensemble_save_path}")

# Example: Make predictions on sample data
print("\n" + "="*70)
print("PREDICTION EXAMPLE ON TEST DATA")
print("="*70)

# Display predictions for sample test cases
sample_indices = np.random.choice(len(X_test), size=5, replace=False)

prediction_results = []
for idx in sample_indices:
    sample = X_test.iloc[idx:idx+1]
    actual = y_test.iloc[idx]
    pred = stacking_ensemble.predict(sample)[0]
    confidence = stacking_ensemble.predict_proba(sample)[0][1]  # probability of viral
    
    prediction_results.append({
        'Actual': 'Viral' if actual == 1 else 'Not Viral',
        'Predicted': 'Viral' if pred == 1 else 'Not Viral',
        'Virality Confidence': f'{confidence:.3f}',
        'Correct': '✓' if actual == pred else '✗'
    })

pred_df = pd.DataFrame(prediction_results)
print("\nSample Predictions:")
print(pred_df.to_string(index=False))

# Summary statistics
print("\n" + "="*70)
print("ENSEMBLE SUMMARY STATISTICS")
print("="*70)
print(f"Best Metric: {results_df.loc[0, 'Model']}")
print(f"  - Accuracy:  {results_df.loc[0, 'Accuracy']:.4f}")
print(f"  - ROC-AUC:   {results_df.loc[0, 'ROC-AUC']:.4f}")
print(f"  - F1-Score:  {results_df.loc[0, 'F1-Score']:.4f}")
print(f"\nTotal Test Samples: {len(y_test)}")
print(f"Correctly Classified: {(ensemble_pred == y_test).sum()}")
print(f"Misclassified: {(ensemble_pred != y_test).sum()}")
print("="*70)
